# Candidate generation and scoring

Set eval_input to a list of question-level prepared/raw files and model_ref to a list of model IDs, adapter directories or registry aliases. Generate training preferences only from training or held-out DPO questions.

sample20, sample10 and sample5 differ only in sample count. literal_copy changes the answer prompt. Keep the source, prompt and seed the same when comparing models. Each model writes its own candidate bank.

merge_models takes banks mapping labels to JSONL paths and prefixes response IDs with provenance. filter takes input_jsonl and sample/parser filters. score_logprobs accepts an optional reference_model_ref. No model is loaded during preview.

Execution is **off by default**. Preview needs only Python and the repository. Select a kernel with the project's runtime dependencies before executing. Each method uses that kernel's Python in a separate worker process.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


Project: C:\Users\Dustin\Projects\BioASQ-MRES
Python: c:\Users\Dustin\Projects\BioASQ-MRES\.venv\Scripts\python.exe


## Choose a preset

- **sample20**: sample_candidates
- **sample10**: sample_candidates
- **sample5**: sample_candidates
- **literal_copy**: sample_candidates
- **score_logprobs**: score_candidates
- **filter**: filter_candidates
- **merge_models**: merge_candidates

Edit the configuration, then rerun the remaining cells. OVERRIDES uses the option names printed by describe(), with underscores. For CLI flags, True includes the named flag and False omits it. Missing inputs block execution.


In [2]:
CATEGORY = 'generation'
PRESET = 'sample10'

DATA = (
    PROJECT_ROOT
    / "data/BioASQ_factoid_sft_prepared"
    / "split_first_train90_dev10_supported_train_seed3407"
)
SFT_RUNS = PROJECT_ROOT / "Artifacts/notebook_runs/sft/answer_sft"

OVERRIDES = {
    # Unique supported training questions: 1,137 questions x 10 samples.
    "eval_input": [str(DATA / "train_questions.json")],
    "samples_per_question_total": 10,
    "temperature": 0.7,
    "top_p": 0.9,
    "max_new_tokens": 64,
    "batch_size": 1,
}
RUN = True
ALLOW_API = False

# Run in separate worker processes so GPU memory is released between models.
VARIANTS = [
    {
        "label": "qwen25_05b_sft_sample10_train",
        "overrides": {
            "model_ref": [str(SFT_RUNS / "20260927-143545-67f7a338/adapter")],
        },
    },
    {
        "label": "qwen25_3b_sft_sample10_train",
        "overrides": {
            "model_ref": [str(SFT_RUNS / "20260927-143545-3fca1c3d/adapter")],
        },
    },
]

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))


{
  "method": "sample_candidates",
  "parameters": {
    "chat_template": "qwen-2.5",
    "prompt_format": "chat",
    "prompt_file": "prompts/factoid_single_answer_aligned.json",
    "prompt": "factoid-single-answer-extractive-v1",
    "max_resources": 0,
    "max_resource_chars": 0,
    "max_seq_length": 4096,
    "eval_input": [
      "C:\\Users\\Dustin\\Projects\\BioASQ-MRES\\data\\BioASQ_factoid_sft_prepared\\split_first_train90_dev10_supported_train_seed3407\\train_questions.json"
    ],
    "model_ref": null,
    "question_types": [
      "factoid"
    ],
    "samples_per_question_total": 10,
    "temperature": 0.7,
    "top_p": 0.9,
    "max_new_tokens": 64,
    "seed": 3407,
    "batch_size": 1
  }
}


## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [3]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


{
  "method": "sample_candidates",
  "description": "Sample one or more models using a shared prompt and source split.",
  "defaults": {
    "chat_template": "qwen-2.5",
    "prompt_format": "chat",
    "prompt_file": "prompts/factoid_single_answer_aligned.json",
    "prompt": "factoid-single-answer-extractive-v1",
    "max_resources": 0,
    "max_resource_chars": 0,
    "max_seq_length": 4096,
    "eval_input": null,
    "model_ref": null,
    "question_types": [
      "factoid"
    ],
    "samples_per_question_total": 20,
    "temperature": 0.7,
    "top_p": 0.9,
    "max_new_tokens": 64,
    "seed": 3407,
    "batch_size": 1
  },
  "required": [
    "eval_input",
    "model_ref"
  ],
  "options": {
    "eval_input": {
      "nargs": "+",
      "required": true,
      "help": "Raw BioASQ JSON or prepared JSON files containing training questions.",
      "flag": "--eval-input"
    },
    "model_ref": {
      "nargs": "+",
      "default": null,
      "help": "Model registry alias/run 

## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [4]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.
W0928 11:50:49.619052 25068 .venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0928 11:50:49.715292 25068 .venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
🦥 Unsloth Zoo will now patch everything to make training faster!
c:\Users\Dustin\Projects\BioASQ-MRES\.venv\lib\site-packages\unsloth\import_fixes.p

## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [5]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])



Run: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\generation\sample_candidates\20260928-015015-c74d47ec
{
  "status": "complete",
  "method": "sample_candidates",
  "fingerprint": "df81af6662286404cbc040dacb45339e1fe89439c7df0dfa9ad58d8fb599b4ed"
}

Log: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\generation\sample_candidates\20260928-015015-c74d47ec\run.log
Outputs: ['candidates', 'status.json']

Run: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\generation\sample_candidates\20260928-015015-cbfc5910
{
  "status": "complete",
  "method": "sample_candidates",
  "fingerprint": "222f2b2cc60446e4b6f6c8d14ed507c5274726175c0d8be838191b5a11a6b2fe"
}

Log: C:\Users\Dustin\Projects\BioASQ-MRES\Artifacts\notebook_runs\generation\sample_candidates\20260928-015015-cbfc5910\run.log
Outputs: ['candidates', 'status.json']
